# M-κ Analysis - Refactored Implementation

Testing the new clean MKappaAnalysis class that uses StressStrainProfile.

## Approach

1. Create cross-section with geometry, concrete, and reinforcement
2. Initialize MKappaAnalysis
3. Solve for M-κ relationship
4. Visualize results
5. Validate against expected behavior

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Import refactored modules
from bmcs_cross_section.cs_design.shapes import RectangularShape
from bmcs_cross_section.cs_design.reinforcement import ReinforcementLayout, ReinforcementLayer
from bmcs_cross_section.cs_design.cross_section import CrossSection
from bmcs_cross_section.matmod.ec2_concrete import EC2Concrete
from bmcs_cross_section.matmod.steel_reinforcement import SteelReinforcement
from bmcs_cross_section.mkappa.mkappa import MKappaAnalysis, create_default_mkappa

print("Imports successful!")

## Test 1: Default Cross-Section

Use the convenience function to create a standard rectangular section.

In [ ]:
# Create default M-κ analysis
mkappa = create_default_mkappa()

print(f"Cross-section: {mkappa.cs.shape.b} × {mkappa.cs.shape.h} mm")
print(f"Concrete: f_ck = {mkappa.cs.concrete.f_ck} MPa")
print(f"Reinforcement layers: {len(mkappa.cs.reinforcement.layers)}")
for i, layer in enumerate(mkappa.cs.reinforcement.layers):
    print(f"  Layer {i+1}: z = {layer.z} mm, A_s = {layer.A_s:.1f} mm²")

## Test 2: Determine Curvature Range

Check the automatically determined curvature range based on material limits.

In [ ]:
kappa_min, kappa_max = mkappa.get_kappa_range()

print(f"Curvature range:")
print(f"  κ_min = {kappa_min:.6f} mm⁻¹ = {kappa_min*1000:.4f} ‰/mm")
print(f"  κ_max = {kappa_max:.6f} mm⁻¹ = {kappa_max*1000:.4f} ‰/mm")
print(f"\nMaterial limits:")
print(f"  ε_cu = {mkappa.cs.concrete.eps_cu1:.4f} (concrete ultimate compression)")
print(f"  h = {mkappa.cs.h_total:.1f} mm")

## Test 3: Solve Single Equilibrium Point

Test equilibrium solver at a specific curvature.

In [ ]:
# Test at mid-range curvature
kappa_test = (kappa_min + kappa_max) / 2.0

print(f"Testing equilibrium at κ = {kappa_test:.6f} mm⁻¹")

eps_bot, N, M, converged = mkappa.solve_equilibrium_at_kappa(kappa_test)

print(f"\nSolution:")
print(f"  ε_bot = {eps_bot:.6f}")
print(f"  N = {N:.3f} kN (should be ≈ 0)")
print(f"  M = {M:.3f} kN·m")
print(f"  Converged: {converged}")

# Check if equilibrium is satisfied
if converged and abs(N) < mkappa.N_tol:
    print(f"\n✓ Equilibrium satisfied (|N| < {mkappa.N_tol} kN)")
else:
    print(f"\n✗ Equilibrium NOT satisfied (|N| = {abs(N):.3f} kN)")

## Test 4: Solve Complete M-κ Curve

Compute the full moment-curvature relationship.

In [ ]:
# Solve for all curvature points
print("Solving M-κ relationship...")
mkappa.solve()
print("Done!")

print(f"\nSolution summary:")
print(f"  Points computed: {len(mkappa.kappa_values)}")
print(f"  κ range: [{mkappa.kappa_values[0]*1000:.4f}, {mkappa.kappa_values[-1]*1000:.4f}] ‰/mm")
print(f"  M range: [{mkappa.M_values.min():.2f}, {mkappa.M_values.max():.2f}] kN·m")
print(f"  Max |N| residual: {np.abs(mkappa.N_values).max():.4f} kN")

## Test 5: Plot M-κ Curve

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
mkappa.plot_mk(ax=ax)
plt.tight_layout()
plt.show()

## Test 6: Plot State at Specific Curvatures

Visualize strain and stress distributions at key points.

In [ ]:
# Plot at 25%, 50%, and 75% of max curvature
kappa_percentages = [0.25, 0.50, 0.75]
kappa_max_actual = mkappa.kappa_values[-1]

for pct in kappa_percentages:
    kappa_target = kappa_max_actual * pct
    
    fig, (ax_strain, ax_stress) = plt.subplots(1, 2, figsize=(14, 8))
    mkappa.plot_state_at_kappa(kappa_target, ax_strain, ax_stress, show_resultants=True)
    plt.tight_layout()
    plt.show()

## Test 7: Check Equilibrium Quality

Verify that axial force is close to zero at all points.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(mkappa.kappa_values * 1000, mkappa.N_values, 'o-', markersize=3)
ax.axhline(0, color='red', linestyle='--', linewidth=1, label='N = 0')
ax.axhline(mkappa.N_tol, color='orange', linestyle=':', linewidth=1, label=f'Tolerance = ±{mkappa.N_tol} kN')
ax.axhline(-mkappa.N_tol, color='orange', linestyle=':', linewidth=1)

ax.set_xlabel(r'Curvature $\kappa$ [‰/mm]', fontsize=12)
ax.set_ylabel(r'Axial Force $N$ [kN]', fontsize=12)
ax.set_title('Equilibrium Quality Check', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, linestyle='--')
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()

print(f"Equilibrium statistics:")
print(f"  Mean |N|: {np.abs(mkappa.N_values).mean():.4f} kN")
print(f"  Max |N|: {np.abs(mkappa.N_values).max():.4f} kN")
print(f"  Std |N|: {np.abs(mkappa.N_values).std():.4f} kN")

## Test 8: Custom Cross-Section

Create a different cross-section and compare results.

In [ ]:
# Create 400×600 mm section with higher strength concrete
shape2 = RectangularShape(b=400.0, h=600.0)
concrete2 = EC2Concrete(f_ck=50.0)  # C50/60
steel_mat2 = SteelReinforcement(f_sy=500.0)

# Heavier reinforcement
layer_bottom2 = ReinforcementLayer(
    z=60.0,
    A_s=2513.3,  # 8Ø20
    material=steel_mat2
)

layer_top2 = ReinforcementLayer(
    z=540.0,
    A_s=804.2,   # 4Ø16
    material=steel_mat2
)

reinforcement2 = ReinforcementLayout(layers=[layer_bottom2, layer_top2])

cs2 = CrossSection(
    shape=shape2,
    concrete=concrete2,
    reinforcement=reinforcement2
)

mkappa2 = MKappaAnalysis(cs=cs2, n_kappa=100)

print("Solving second cross-section...")
mkappa2.solve()
print("Done!")

print(f"\nComparison:")
print(f"  Section 1 (300×500, C30): M_max = {mkappa.M_values.max():.2f} kN·m")
print(f"  Section 2 (400×600, C50): M_max = {mkappa2.M_values.max():.2f} kN·m")

## Test 9: Compare M-κ Curves

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

mkappa.plot_mk(ax=ax, color='#1f77b4', linewidth=2.0)
ax.lines[-1].set_label('300×500 mm, C30/37, 4Ø20')

mkappa2.plot_mk(ax=ax, color='#ff7f0e', linewidth=2.0)
ax.lines[-1].set_label('400×600 mm, C50/60, 8Ø20')

ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

## Test 10: Validation Summary

Check that results make physical sense.

In [ ]:
print("=" * 60)
print("VALIDATION SUMMARY")
print("=" * 60)

def validate_mkappa(mk, name):
    print(f"\n{name}:")
    
    # Check 1: M increases with κ (initially)
    dM_dkappa = np.diff(mk.M_values)
    n_positive = np.sum(dM_dkappa > 0)
    print(f"  ✓ M increases initially: {n_positive}/{len(dM_dkappa)} points")
    
    # Check 2: Peak moment exists
    M_max = mk.M_values.max()
    idx_max = np.argmax(mk.M_values)
    kappa_at_M_max = mk.kappa_values[idx_max]
    print(f"  ✓ Peak moment: M_max = {M_max:.2f} kN·m at κ = {kappa_at_M_max*1000:.4f} ‰/mm")
    
    # Check 3: Equilibrium satisfied
    max_N_error = np.abs(mk.N_values).max()
    print(f"  ✓ Equilibrium: max |N| = {max_N_error:.4f} kN")
    
    # Check 4: Reasonable moment values
    h = mk.cs.h_total
    b = mk.cs.shape.b
    f_c = mk.cs.concrete.f_ck
    M_approx = 0.1 * b * h**2 * f_c / 1e6  # Very rough estimate
    print(f"  ✓ Moment magnitude reasonable: M_max = {M_max:.2f} kN·m (rough estimate: {M_approx:.2f} kN·m)")
    
    return True

validate_mkappa(mkappa, "Section 1")
validate_mkappa(mkappa2, "Section 2")

print("\n" + "=" * 60)
print("All tests completed!")
print("=" * 60)

In [ ]:
# Set to a specific curvature near peak moment
idx_max = np.argmax(mkappa.M_values)
kappa_at_peak = mkappa.kappa_values[idx_max]

print(f"Setting state to peak moment:")
print(f"  κ = {kappa_at_peak*1000:.4f} ‰/mm")
print(f"  M = {mkappa.M_values[idx_max]:.2f} kN·m")

mkappa_state.set_kappa(kappa_at_peak)
fig, ax_strain, ax_stress, ax_mk = mkappa_state.plot_mkappa_state(figsize=(18, 6))
plt.show()

## Test 13: Update State Interactively

Test the ability to change the curvature and update the visualization.

In [ ]:
# Plot states at 25%, 50%, 75%, and 100% of maximum curvature
print("Generating state visualizations at key points...")
mkappa_state.plot_at_percentages(
    percentages=[0.25, 0.50, 0.75, 1.0],
    figsize=(18, 6),
    show_resultants=True
)

## Test 12: Explore Different States

Visualize the cross-section state at different points along the M-κ curve.

In [ ]:
from bmcs_cross_section.mkappa.mkappa_state_profiles import MKappaStateProfiles

# Create state visualization at mid-point of M-κ curve
mkappa_state = MKappaStateProfiles(mkappa=mkappa)

print(f"Current state:")
print(f"  κ = {mkappa_state.kappa*1000:.4f} ‰/mm")
print(f"  M = {mkappa_state._get_M_at_kappa(mkappa_state.kappa):.2f} kN·m")
print("\nPlotting combined visualization...")

fig, ax_strain, ax_stress, ax_mk = mkappa_state.plot_mkappa_state(figsize=(18, 6))
plt.show()

## Test 11: Interactive M-κ State Visualization

Test the new MKappaStateProfiles class that combines M-κ curve with stress-strain profiles.